# BS4 실패 URL Selenium 재시도 (로컬용)

`본문_bs4_재실패_*.json`에 들어 있는 실패 URL만 Selenium으로 다시 열어 본다. 성공한 기사는 별도 CSV로 저장하고, 필요하면 기존 `본문_bs4_*.csv`에 병합한다.

- 입력: `data/본문_bs4_재실패_*.json`
- 출력: `data/본문_selenium_재시도성공_*.csv`, `data/본문_selenium_재시도실패_*.json`
- 특징: 실패 URL만 재시도, 브라우저 화면 확인 가능, 기존 CSV 백업 후 병합


In [ ]:
# # 필요한 패키지 설치
# # 로컬 커널에 selenium/pandas가 이미 있으면 이 셀은 건너뛰어도 됨
# %pip install -q selenium pandas


In [ ]:
from pathlib import Path
import json
import os
import platform
import random
import shutil
import subprocess
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 로컬/Colab 결과 차이를 줄이기 위해 URL 수집 때와 같은 User-Agent 사용
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 브라우저를 직접 보면서 확인하려면 False 유지, 창 없이 돌리려면 True로 변경
# WSL에서 GUI 디스플레이가 없으면 True로 두는 편이 안정적
HEADLESS = False

# 재시도할 때 서버에 너무 촘촘히 붙지 않도록 대기
RETRY_PAUSE_RANGE_SEC = (1.0, 2.5)
# driver.get 직후 DOM 안정화 시간 — JS 렌더링 늦은 페이지 대응
PAGE_LOAD_WAIT_SEC = 0.8
# WebDriverWait 최대 대기 — 2차 확인 시 본문/제목 요소 등장까지 기다리는 한계
SELENIUM_WAIT_SEC = 8

# 성공분을 기존 본문_bs4 CSV에 바로 합치기 (False면 별도 성공 CSV만 떨어뜨림)
MERGE_TO_BS4_CSV = True

# 병합 전 기존 CSV 백업 만들기 (.bak_before_selenium_retry 접미사)
MAKE_BACKUP = True

# 재시도 대상 실패 파일 지정
# None이면 data 폴더의 모든 본문_bs4_재실패_*.json 재시도
# 특정 파일만 하려면 예: TARGET_FAILURE_FILES = ['본문_bs4_재실패_SBS_260505_260505.json']
TARGET_FAILURE_FILES = None

# === 기존 자동 탐색 (Colab 또는 Drive 동기화 환경용) — 주석처리 ===
# NOTEBOOK_DIR = Path.cwd()
# if NOTEBOOK_DIR.name != 'crawling':
#     candidate = Path('/home/carol/Text-data-Analysis_26-Spring/news/notebook/crawling')
#     NOTEBOOK_DIR = candidate if candidate.exists() else NOTEBOOK_DIR
# DATA_DIR = NOTEBOOK_DIR / 'data'

# === 현재 로컬 작업 폴더에 직접 지정 ===
# news/ 폴더 바로 안에 본문_bs4_재실패_*.json / 본문_bs4_*.csv 가 모두 들어있음
DATA_DIR = Path('/home/carol/Text-data-Analysis_26-Spring/news')

print(f'DATA_DIR: {DATA_DIR}')
print(f'User-Agent: {USER_AGENT}')


In [ ]:
# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range=RETRY_PAUSE_RANGE_SEC):
    pause_sec = random.uniform(*pause_range)
    print(f'{label} {pause_sec:.1f}초 대기')
    time.sleep(pause_sec)


# 로컬 OS에 설치된 Chrome 실행 파일 탐색
# Selenium Manager 기본 탐색이 종종 실패해서 직접 binary_location을 지정하는 편이 안정적
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        # Windows는 Chrome 설치 경로가 정해져 있어 환경 변수 기반으로 후보 나열
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        # 리눅스/WSL/macOS는 PATH에서 검색 (chrome / chromium 호환 이름 4종)
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)

    # 후보 중 실제로 존재하는 첫 번째 경로 반환, 모두 실패하면 None
    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# Selenium 드라이버 생성
# 실패 URL을 눈으로 확인할 수 있도록 기본값은 브라우저 창을 띄우는 방식
def build_driver(headless=HEADLESS):
    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 수집 환경 고정
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동화 제어 관련 switch 제외
    options.add_experimental_option('useAutomationExtension', False)  # Selenium 자동화 확장 비활성화
    options.add_argument('--disable-blink-features=AutomationControlled')  # AutomationControlled 플래그 비활성화
    options.add_argument('--window-size=1400,1000')  # 일정한 화면 크기로 렌더링

    if headless:
        options.add_argument('--headless=new')  # 창 없이 실행할 때 새 headless 모드 사용

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스 컨테이너/WSL 환경에서 Chrome 실행 안정화
        options.add_argument('--disable-dev-shm-usage')  # /dev/shm 용량 부족으로 Chrome이 죽는 문제 완화
        options.add_argument('--disable-gpu')  # headless 환경에서 GPU 관련 오류 방지

    # Chrome 실행 파일을 찾으면 명시하고, 못 찾으면 Selenium Manager 기본 탐색 사용
    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'Chrome binary: {chrome_binary}')
        subprocess.run([chrome_binary, '--version'], check=False)
    else:
        print('Chrome binary를 직접 찾지 못함 — Selenium Manager 기본 탐색 사용')

    # Selenium Manager가 현재 Chrome 버전에 맞는 ChromeDriver를 자동으로 찾거나 내려받음
    driver = webdriver.Chrome(service=Service(), options=options)

    # navigator.webdriver 플래그 제거 — 자동화 탐지를 회피하기 위해 새 페이지 진입 시마다 주입
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


driver = build_driver()


In [ ]:
import re


# 일반 뉴스 URL이 리다이렉트되는 서브포털 도메인 매핑
# 키: 호스트 substring 검사용 / 값: (카테고리 prefix, URL path 첫 segment 추출 정규식)
# 도메인이 추가되면 여기에만 등록하면 분기 함수가 자동으로 처리
REDIRECT_DOMAINS = {
    'sports.naver.com': ('스포츠', r'https://m\.sports\.naver\.com/([^/]+)/article/'),
    'entertain.naver.com': ('연예', r'https://m\.entertain\.naver\.com/([^/]+)/article/'),
}


# 한글 표시 시각을 일반 뉴스의 data-date-time과 같은 포맷으로 변환
# 예: '2026.05.05. 오전 7:10' -> '2026-05-05 07:10:00'
def parse_korean_datetime(text):
    m = re.match(r'(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d{1,2}):(\d{2})', text)
    if not m:
        return ''
    y, mo, d, ampm, h, mi = m.groups()
    h = int(h)
    # 12시간제 → 24시간제 변환 (오전 12시 = 00시, 오후 12시 = 12시 그대로)
    if ampm == '오후' and h != 12:
        h += 12
    elif ampm == '오전' and h == 12:
        h = 0
    return f'{y}-{mo}-{d} {h:02d}:{mi}:00'


# 네이버 서브포털(스포츠/연예)은 일반 뉴스와 HTML 구조가 달라 별 셀렉터 사용
# 두 도메인 모두 Next.js 기반이라 og:title / div._article_content / em.date 셀렉터를 공유
def extract_redirected_article_selenium(driver, original_link):
    current_url = driver.current_url

    # 제목 추출하기 — meta og:title 사용 (Next.js CSS 모듈 hash가 빌드마다 바뀌어 더 안정적)
    title_elements = driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:title"]')
    title = title_elements[0].get_attribute('content').strip() if title_elements else ''

    # 본문 추출하기 — div._article_content (언더스코어 prefix는 hash에 의존하지 않음)
    body_elements = driver.find_elements(By.CSS_SELECTOR, 'div._article_content')
    body = body_elements[0].text.replace('\n', '').strip() if body_elements else ''

    # 날짜 추출하기 — em.date 첫 번째(입력일). 두 번째는 수정일이라 무시
    date_elements = driver.find_elements(By.CSS_SELECTOR, 'em.date')
    pubdate = parse_korean_datetime(date_elements[0].text.strip()) if date_elements else ''

    # 카테고리 추출하기 — 도메인 매핑 dict에서 prefix를 찾고 URL path 첫 segment를 붙임
    # 예: m.sports.naver.com/golf/article/... -> '스포츠/golf'
    category = '기타'
    for domain, (prefix, pattern) in REDIRECT_DOMAINS.items():
        if domain in current_url:
            cat_match = re.match(pattern, current_url)
            category = f'{prefix}/{cat_match.group(1)}' if cat_match else prefix
            break

    # 제목/본문/날짜 중 하나라도 없으면 실패로 기록
    if not title or not body or not pubdate:
        raise ValueError(f'리다이렉트 기사 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    # 호출부에서 원래 URL을 키로 쓰므로 link 필드는 원본 URL로 유지
    return {
        'link': original_link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 기사 한 건에서 title/body/pubdate/category 추출 (Selenium 버전)
# 본문_수집.ipynb의 extract_article과 동일한 셀렉터를 쓰되, 한 번 더 WebDriverWait로 재확인하는 보강 단계가 있음
def extract_article_selenium(driver, link):
    # 실제 네이버 뉴스 웹페이지로 이동
    driver.get(link)

    # 페이지 로딩 대기
    wait = WebDriverWait(driver, SELENIUM_WAIT_SEC)
    time.sleep(PAGE_LOAD_WAIT_SEC)

    # 리다이렉트 감지 — 일반 뉴스 URL이 스포츠/연예 서브포털로 빠지면 별 추출기로 분기
    if any(domain in driver.current_url for domain in REDIRECT_DOMAINS):
        return extract_redirected_article_selenium(driver, link)

    # 제목 추출하기
    title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
    title = title_elements[0].text.strip() if title_elements else ''

    # 본문 추출하기 — 줄바꿈 제거해서 단일 문자열로 정리
    body_elements = driver.find_elements(By.ID, 'newsct_article')
    body = body_elements[0].text.replace('\n', '').strip() if body_elements else ''

    # 날짜 추출하기 — 사람이 읽는 라벨이 아니라 data-date-time 속성값을 사용 (ISO 포맷)
    pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else ''

    # 카테고리 추출하기 — 페이지 상단 카테고리 탭 중 현재 활성화된(aria-selected="true") 항목
    category_elements = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
    category = category_elements[0].text.strip() if category_elements else ''

    if not title or not body or not pubdate:
        # 아주 짧게 한 번 더 기다린 뒤 재확인 — BS4가 놓친 늦은 렌더링 케이스 보강
        wait.until(lambda d: d.find_elements(By.ID, 'newsct_article') or d.find_elements(By.CLASS_NAME, 'media_end_head_headline'))
        title_elements = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        body_elements = driver.find_elements(By.ID, 'newsct_article')
        pubdate_elements = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')

        # 재확인에서 얻은 값으로 덮어쓰되, 여전히 비어있으면 이전 값을 그대로 유지
        title = title_elements[0].text.strip() if title_elements else title
        body = body_elements[0].text.replace('\n', '').strip() if body_elements else body
        pubdate = pubdate_elements[0].get_attribute('data-date-time') if pubdate_elements else pubdate

    # 제목/본문/날짜 중 하나라도 없으면 실패로 기록 — 어느 필드가 비었는지 함께 표시
    if not title or not body or not pubdate:
        raise ValueError(f'title/body/pubdate 추출 실패: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


In [ ]:
# 재시도 대상 실패 파일 로드
# BS4 수집 단계에서 떨어뜨린 본문_bs4_재실패_*.json 목록을 추려 반환
def load_failure_files(data_dir=DATA_DIR):
    # TARGET_FAILURE_FILES가 지정되면 해당 파일만, 아니면 모든 재실패 파일 처리
    if TARGET_FAILURE_FILES:
        paths = [data_dir / name for name in TARGET_FAILURE_FILES]
    else:
        paths = sorted(data_dir.glob('본문_bs4_재실패_*.json'))

    # 존재 여부 확인 — 누락 파일은 경고만 출력하고 계속 진행
    existing = [p for p in paths if p.exists()]
    missing = [p for p in paths if not p.exists()]

    if missing:
        print('찾지 못한 실패 파일:')
        for p in missing:
            print(f'- {p}')

    if not existing:
        raise FileNotFoundError('재시도할 본문_bs4_재실패_*.json 파일이 없습니다.')

    # 파일별 실패 건수 미리 출력 — 재시도 분량 사전 확인용
    print(f'재시도 대상 실패 파일: {len(existing)}개')
    for p in existing:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'- {p.name}: {len(data.get("links", []))}건')
    return existing


failure_files = load_failure_files()


In [ ]:
# 실패 URL만 Selenium으로 다시 방문해 본문 수집 재시도
# 성공분은 별도 CSV로 저장하고 옵션에 따라 기존 BS4 CSV에 병합
def retry_failure_file(failure_path, driver=driver, data_dir=DATA_DIR):
    # 파일명 키(예: SBS_260505_260505) 추출 후 관련 경로 일괄 생성
    stem = failure_path.stem.replace('본문_bs4_재실패_', '')
    bs4_csv_path = data_dir / f'본문_bs4_{stem}.csv'
    success_path = data_dir / f'본문_selenium_재시도성공_{stem}.csv'
    remain_fail_path = data_dir / f'본문_selenium_재시도실패_{stem}.json'

    # 재실패 JSON에서 원래 index와 URL 목록 불러오기
    with failure_path.open('r', encoding='utf-8') as f:
        failure_data = json.load(f)

    failed_indices = failure_data.get('err_idx', [])
    failed_links = failure_data.get('links', [])

    successes = []
    failures = []

    print()
    print(f'=== {stem} Selenium 재시도 시작: {len(failed_links)}건 ===')

    # 실패 URL을 하나씩 Selenium으로 재방문
    # original_idx는 원본 BS4 수집 시점의 인덱스로 추적용 메타로 보존
    for seq, (original_idx, link) in enumerate(zip(failed_indices, failed_links), start=1):
        try:
            article = extract_article_selenium(driver, link)
            article['original_idx'] = original_idx
            successes.append(article)
            print(f'성공 [{seq}/{len(failed_links)}] index={original_idx}')
        except Exception as exc:
            # 실패 사유까지 같이 보존 — 디버깅/후속 분석용
            failures.append({'original_idx': original_idx, 'link': link, 'error': repr(exc)})
            print(f'실패 [{seq}/{len(failed_links)}] index={original_idx}: {exc!r}')

        polite_sleep('다음 재시도 전')

    # Selenium 재시도 성공분 저장 (성공 0건이면 CSV 생성 생략)
    success_df = pd.DataFrame(successes)
    if not success_df.empty:
        success_df.to_csv(success_path, index=False, encoding='utf-8-sig')
        print(f'Selenium 성공분 저장: {success_path}')

    # 재시도 후에도 실패한 URL과 오류 이유 저장 (성공/실패 무관하게 항상 기록 — 0건이어도 트레이스 보존)
    with remain_fail_path.open('w', encoding='utf-8') as f:
        json.dump({'failures': failures}, f, ensure_ascii=False, indent=2)
    print(f'Selenium 재시도 실패 목록 저장: {remain_fail_path}')

    # 성공분을 기존 BS4 CSV에 병합
    if MERGE_TO_BS4_CSV and not success_df.empty:
        if not bs4_csv_path.exists():
            raise FileNotFoundError(f'병합할 기존 CSV가 없습니다: {bs4_csv_path}')

        # 기존 CSV와 성공분을 합친 뒤 링크 기준 중복 제거, 날짜순 정렬
        original_df = pd.read_csv(bs4_csv_path, encoding='utf-8-sig')
        # 성공분에서 원본 스키마(link/pubdate/category/title/body)만 골라 합침 — original_idx는 병합 결과에 포함하지 않음
        merge_df = success_df[['link', 'pubdate', 'category', 'title', 'body']].copy()
        merged_df = pd.concat([original_df, merge_df], ignore_index=True)
        # keep='last'로 같은 link가 있을 때 Selenium 재시도분(뒤쪽)이 원본을 덮어쓰도록 설정
        merged_df = merged_df.drop_duplicates(subset=['link'], keep='last').reset_index(drop=True)
        # pubdate를 datetime으로 변환 — 정렬용이지만 컬럼 dtype 자체가 datetime64로 바뀌는 side effect 있음
        merged_df['pubdate'] = pd.to_datetime(merged_df['pubdate'], errors='coerce')
        merged_df = merged_df.sort_values('pubdate').reset_index(drop=True)

        # 병합 전 백업 — .bak_before_selenium_retry로 1회만 생성, 재실행해도 덮어쓰지 않음
        if MAKE_BACKUP:
            backup_path = bs4_csv_path.with_suffix('.csv.bak_before_selenium_retry')
            if not backup_path.exists():
                shutil.copy2(bs4_csv_path, backup_path)
                print(f'기존 CSV 백업: {backup_path}')

        # 병합본을 기존 BS4 CSV 위치로 덮어쓰기 — 별도 파일 만들지 않고 원본을 갱신
        merged_df.to_csv(bs4_csv_path, index=False, encoding='utf-8-sig')
        print(f'기존 CSV 병합 완료: {bs4_csv_path}')
        print(f'행 수: {len(original_df)} -> {len(merged_df)}')

    print(f'완료: 성공 {len(successes)}건 / 실패 {len(failures)}건')
    return {
        'period': stem,
        'success': len(successes),
        'fail': len(failures),
        'success_path': str(success_path),
        'remain_fail_path': str(remain_fail_path),
    }


In [ ]:
# 생성된 failure_files를 순서대로 재시도
# 한 파일이 실패해도 결과를 요약할 수 있게 파일별 결과 저장
retry_results = []
for failure_path in failure_files:
    result = retry_failure_file(failure_path)
    retry_results.append(result)

# 파일별 성공/실패 건수를 표로 정리 — 노트북 출력에 그대로 표시
summary_df = pd.DataFrame(retry_results)
summary_df


In [ ]:
# 작업이 끝나면 브라우저 종료
# 필요하면 이 셀만 따로 실행
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')
